# Teacher Preview on GPU

Run a larger teacher preview on a rented GPU. This notebook generates teacher outputs for a small batch, saves progress after each sample, and reports `is_correct` / `is_usable`.

## 1. Configure Run

Set `MODEL_NAME` to a Hugging Face repo id or a local model path on the GPU machine.

In [ ]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
NUM_SAMPLES = 100
MAX_NEW_TOKENS = 1024
MAX_RETRIES = 2
TEMPERATURE = 0.0

INPUT_PATH = "data/gsm8k_clean_train.jsonl"
OUTPUT_PATH = "data/gsm8k_teacher_preview_100.jsonl"

## 2. Imports

In [ ]:
import json
import sys
from pathlib import Path

from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.generate_teacher_preview import (
    build_retry_prompt,
    build_teacher_record,
    read_jsonl,
    write_jsonl,
)
from src.teacher.local_teacher import generate_teacher_output, load_local_teacher
from src.utils.prompts import build_strategy_teacher_prompt

## 3. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 4. Load Data and Model

In [ ]:
examples = read_jsonl(PROJECT_ROOT / INPUT_PATH, limit=NUM_SAMPLES)
print(f"Loaded {len(examples)} examples")

tokenizer, model = load_local_teacher(MODEL_NAME)
print("Teacher model loaded")

## 5. Generate Teacher Preview

This writes after every sample, so partial progress is preserved if the runtime stops.

In [ ]:
output_path = PROJECT_ROOT / OUTPUT_PATH
teacher_records = []

for example in tqdm(examples, desc="Generating teacher traces"):
    prompt = build_strategy_teacher_prompt(example["question"])
    record = None

    for attempt in range(MAX_RETRIES + 1):
        raw_output = generate_teacher_output(
            tokenizer=tokenizer,
            model=model,
            prompt=prompt,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
        )
        record = build_teacher_record(example, raw_output)

        if record["is_format_valid"]:
            break

        prompt = build_retry_prompt(example["question"])

    teacher_records.append(record)
    write_jsonl(teacher_records, output_path)

print(f"Saved {len(teacher_records)} records to {output_path}")

## 6. Summary

In [ ]:
rows = [json.loads(line) for line in output_path.open("r", encoding="utf-8")]
total = len(rows)
correct = sum(row["is_correct"] for row in rows)
format_valid = sum(row["is_format_valid"] for row in rows)
usable = sum(row["is_usable"] for row in rows)

print(f"Total: {total}")
print(f"Correct: {correct}/{total} = {correct / total:.1%}")
print(f"Format valid: {format_valid}/{total} = {format_valid / total:.1%}")
print(f"Usable: {usable}/{total} = {usable / total:.1%}")

failed_ids = [row["id"] for row in rows if not row["is_usable"]]
print("Failed usable ids:", failed_ids[:50])

## 7. Inspect Failures

In [ ]:
for row in rows:
    if row["is_usable"]:
        continue

    print("=" * 100)
    print("id:", row["id"])
    print("ground_truth:", row["ground_truth"])
    print("teacher_answer:", row["teacher_answer"])
    print("is_correct:", row["is_correct"])
    print("is_format_valid:", row["is_format_valid"])
    print("format_checks:", row["format_checks"])
    print("raw_teacher_output preview:")
    print((row["raw_teacher_output"] or "")[:1200])